# 4. Ground-Truthing TwinCalibrator

Part 4 of the tutorial series (`01_Dataset.ipynb`, `02_WFSAndPreprocessing.ipynb`, `03_DeformableMirrorAndClosedLoop.ipynb`). Every real `Tutorials/<Instrument>/Calibrate<Instrument>Twin.ipynb` notebook in this repo fits `AI4AO.TwinCalibrator` to a real, measured bench interaction matrix -- but a real bench's true misregistration, mask distortion, and DM parameters are, by definition, unknown, so there's no way to check whether the fit actually recovered the right answer, only whether its residual went down.

This notebook builds an entirely synthetic "fake bench" instead: a "true" twin (`PyramidWFS` + `DeformableMirror`) with deliberately injected, precisely known parameter values, and a "nominal" twin that starts from `PyramidWFS`/`DeformableMirror`'s plain library defaults. We generate the true twin's interaction matrix as a stand-in for a real bench measurement, then run `TwinCalibrator` to fit the nominal twin to it -- and, unlike every real calibration notebook, we can directly compare the *recovered* parameter values against the *injected* ground truth to see whether the fit actually worked, not just whether the loss decreased. The calibrated twin this notebook saves at the end is what `05_TrainingAReconstructor.ipynb` then loads to train a reconstructor on.

## What `TwinCalibrator` can and can't fit

`fit_dm_and_offsets` only ever puts `dm.parameters()` and `wfs.parameters()` into its optimizer -- concretely, on the DM side that's `rotationAngle`, `grid_shift` (`shiftX`/`shiftY`), `radialScaling`, `tangentialScaling`, `anamorphosisAngle`, `moffatParameter`, `sign` (`signedAmplitude`), and `mechCoupling`; on the WFS side, for a `PyramidWFS`, that's `mainSlope`, `maskShifts`, and `rooftop`. Everything else -- `Nactuator`, `Nmodes`, `FlipLeftRight`/`FlipTopBottom` -- is a plain, non-differentiable Python attribute that determines array shapes and must therefore be **identical** between the true and nominal twins below; only the quantities in the first list are allowed to differ, since those are what the fit is actually trying to recover.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch

from AI4AO import PyramidWFS, DeformableMirror, TwinCalibrator

device = 'cuda'  # set to "cpu" if CUDA is not available

## Shared baseline

Both twins start from the same `wfs_params_exp.py` used throughout this tutorial series. Only `WFSParams` and `DMParams` are needed here -- this is a static calibration exercise, not a closed loop, so `AtmosParams`/`LoopParams` don't come into it. We turn `useNoise` off, matching how the real per-instrument calibration notebooks fit against noise-free/averaged bench data (and `BuildInteractionMatrix`, used below, disables noise internally regardless). We also switch on modulation (`Modulation=3`, vs. the file's default of `0`) and remove the central obstruction (`centralObstruction=0`), so both twins are a standard modulated, unobstructed pyramid rather than the file's default static, obstructed one. The WFS-side dict itself is identical for both twins -- only three of `PyramidWFS`'s parameters (`mainSlope`, `maskShifts`, `rooftop`) get overwritten after construction, and only a copy of `DMParams` differs, for the true twin's DM-parameter overrides.

In [ ]:
paramfile = 'wfs_params_exp.py'

WFSParams = Config.fromfile(paramfile)['WFSParams']
DMParams = Config.fromfile(paramfile)['DMParams']

WFSParams['useNoise'] = False
WFSParams['centralObstruction'] = 0.
WFSParams['Modulation'] = 3.

## The "true" twin: injecting known ground truth

The true twin's `PyramidWFS` gets specific, hand-picked values for its three trainable mask parameters, and its `DeformableMirror` gets a specific misregistration plus a few specific DM-parameter overrides -- including a 61-degree rotation and `FlipLeftRight=True`, both deliberately large/discrete enough that the continuous joint fit alone couldn't find them from a nearby starting point; that's what the `rough_calibrate_dm` step further down is for. There's no dedicated setter for `mainSlope`/`maskShifts`/`rooftop` -- they're plain `nn.Parameter`s, so we overwrite them in place with `.copy_()` under `torch.no_grad()`, exactly as the repo does elsewhere. Calling `.eval()` afterwards rebuilds the mask and reference intensity from the new values automatically (`WFS.train(mode=False)` does this as a side effect), so no separate `BuildMask()` call is needed.

The DM's misregistration is passed straight into the constructor via the `misreg=` dict (the same pattern `Tutorials/Papyrus/CalibrateExamplePapyrusTwin.ipynb` uses), in physical units (degrees, meters, percent of diameter) -- the constructor applies it and rebuilds the actuator grid/zonal modes on its own. `Nactuator`, `Nmodes`, and the flip flags are left untouched from `wfs_params_exp.py`'s `DMParams`, since those must match the nominal twin built later (the `FlipLeftRight=True` override lives on `DMParams_true` specifically, not the shared `DMParams`, so it stays a difference between the two twins rather than becoming another thing that has to match).

In [ ]:
wfs_true = PyramidWFS(WFSParams, device)

with torch.no_grad():
    wfs_true.mainSlope.copy_(torch.tensor(1.4, device=device, dtype=torch.float32))
    wfs_true.rooftop.copy_(torch.tensor(0.95, device=device, dtype=torch.float32))
    wfs_true.maskShifts.copy_(torch.tensor([
        [1.09, 0.90],
        [0.98, 1.04],
        [1.05, 0.99],
        [0.99, 0.91],
    ], device=device, dtype=torch.float32))

wfs_true.eval()  # rebuilds the mask + reference intensity from the new parameter values

true_misreg = dict(
    rotationAngle=61,        # degrees
    shiftX=0.015,              # meters
    shiftY=-0.02,              # meters
    anamorphosisAngle=4.0,     # degrees
    radialScaling=2.0,         # percent of diameter
    tangentialScaling=-1.5,    # percent of diameter
)

DMParams_true = DMParams.copy()
DMParams_true.update(moffatParam=2.6, signedAmplitude=1.3e-5, MechCoupling=0.42, FlipLeftRight=True)

dm_basis = DeformableMirror(WFSParams, DMParams_true, device=device)
nModes = 60
M2C = dm_basis.MakeZernikeM2C(nModes)

dm_true = DeformableMirror(WFSParams, DMParams_true, device=device, misreg=true_misreg)
dm_true.eval()

## Visualizing the true twin

A quick sanity check before generating synthetic bench data: the true twin's reference intensity (what the pyramid mask produces with no aberration) and its actuator grid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(wfs_true.reference_intensity.cpu())
axes[0].set_title("True twin -- reference intensity")
axes[0].axis('off')

axes[1].imshow(dm_true.grid.cpu())
axes[1].set_title(f"True twin -- actuator grid ({dm_true.totalAct.item()} actuators)")
axes[1].axis('off')
plt.show()

## Generating a synthetic "bench" interaction matrix

This uses a *modal* basis rather than a purely zonal one, built from a **third** DM object, `dm_basis`. `dm_basis = DeformableMirror(WFSParams, DMParams_true, device=device)` shares `DMParams_true`'s moffat/sign/mechCoupling values with `dm_true` (so amplitudes/units match), but gets no `misreg` -- it's a nominal, undistorted DM used only to compute `M2C = dm_basis.MakeZernikeM2C(nModes)`, projecting the first 60 Zernike modes onto its influence functions. That's deliberate: a real bench's `M2C` is defined from the instrument's as-designed geometry, not its (unknown, to-be-recovered) as-built misalignment -- computing `M2C` directly from `dm_true` instead would bake the misregistration into the mode shapes themselves, making the recovery exercise circular. `M2C` ends up with 60 columns instead of one per actuator (101, for this `Nactuator`), which is both closer to how a real bench's `M2C` is usually built and cheaper to push through `BuildInteractionMatrix` than the full zonal basis.

`wfs_true.BuildInteractionMatrix` pushes and pulls each of those 60 modes individually (central finite difference) through the true twin's mask and propagator, applying `M2C` to the misregistered `dm_true`, producing exactly the kind of interaction-matrix cube (`Nmodes, Npix, Npix`) a real bench measurement would give -- except here we know precisely what generated it.

In [ ]:
mode_index = list(range(nModes))

modes_true = dm_true(M2C.T)

with torch.no_grad():
    wfs_true.BuildInteractionMatrix(modes_true, pupil=None, batch_size=30, phaseOffset=0)

bench_iMat = wfs_true.iMat.detach().clone()
print(f"Synthetic bench interaction matrix: {tuple(bench_iMat.shape)}")

## The "nominal" twin: starting from library defaults

The nominal twin is what `TwinCalibrator` will actually fit -- it starts from `PyramidWFS`/`DeformableMirror`'s plain, untouched defaults (`mainSlope=pi/2`, `maskShifts=ones(4,2)`, `rooftop=0`, and `misreg=None` i.e. all-zero), mirroring how the real per-instrument notebooks start from as-built geometry before fitting to bench data. `Nactuator`, `Nmodes`, and the flip flags come straight from the same `DMParams` as the true twin, unmodified, since those must match.

We deliberately do not call `.eval()` on this twin, unlike the true twin above -- `nn.Module`s already start in training mode with `requires_grad=True` by construction, and every one of `TwinCalibrator`'s fitting methods calls `.train()` on `wfs`/`dm` at the start of its own loop anyway, so there's nothing to gain from evaluating it first and a real correctness risk in older versions of `AI4AO/TorchPropagator.py`/`DeformableMirror.py` (now fixed) that used to leave gradients permanently frozen after any `.eval()` call.

In [ ]:
wfs_nominal = PyramidWFS(WFSParams, device)
dm_nominal = DeformableMirror(WFSParams, DMParams, device=device)

assert dm_nominal.totalAct == dm_true.totalAct, "true and nominal twins must share the same actuator count"

In [ ]:
calibrator = TwinCalibrator(wfs_nominal, dm_nominal, device)
ref_pupil, ref_phase = calibrator.init_static_offsets()

## First Check

### Visual observation

We can look at the bench and nominal-twin interaction matrices, and their difference, using the `sanity_check_plot` method of the calibrator class. We can observe that the positions of the pupils of the starting guess are incorrect, so we will start by calibrating those.

In [ ]:
calibrator.sanity_check_plot(bench_iMat, M2C, mode_index, idx=4)

### Baseline crosstalk, before any calibration

`rebuild_reconstruction_matrix` recomputes the nominal twin's interaction matrix over the *full* `M2C` (no mode subsetting) and its pseudo-inverse; `crosstalk_diagnostic` then shows how well each bench mode projects onto its own reconstructed mode (a strong diagonal, near-zero off-diagonal is the goal) versus leaking into neighboring modes. Run here, before any fitting at all, this is the "before" picture -- expect it to look close to unstructured noise rather than anything resembling identity, since the nominal twin doesn't resemble the bench yet. Compare it to the same diagnostic run again at the very end of this notebook, after the full calibration.

In [ ]:
modes = calibrator.rebuild_reconstruction_matrix(M2C)
cov = calibrator.crosstalk_diagnostic(bench_iMat)

## Pre-calibrating the mask against the true twin's reference frame

Before attempting the harder joint fit against the interaction matrix below, the real per-instrument notebooks (e.g. `Tutorials/Rama/CalibrateExampleRamaTwin.ipynb`) first align the WFS mask to a measured *reference* frame -- the pyramid's own image with no aberration applied -- via `TwinCalibrator.fit_pupil_to_reference`, before the DM is even involved. We do the same here, using the true twin's `wfs_true.reference_intensity` as a stand-in for that measurement, and alternate it with `TwinCalibrator.fit_rooftop`.

The two calls optimize different things against different losses. `fit_pupil_to_reference(reference_frame, wfs_nominal.parameters(), ...)` fits *all* of the WFS's trainable parameters (`mainSlope`, `maskShifts`, and `rooftop` together) against a plain pixel-wise squared-error loss on the reference frame. `fit_rooftop(reference_frame, ...)` fits *only* `rooftop`, against a different, purpose-built loss that compares the four pupil images' summed flux per quadrant rather than pixel by pixel. `rooftop`'s job is to balance flux between the four pupil images, which a generic pixel-wise loss can be slow to disentangle from `mainSlope`/`maskShifts` when all three share one optimizer -- alternating a few rounds of the dedicated quadrant-flux fit in between rounds of the general fit lets each one clean up what the other is less suited for. Both learning rates anneal down across the 6 outer iterations (`2*10**-(3+i/5)` for the general fit, `10**-(1+i/5)` for `rooftop`), so early rounds make large, coarse corrections and later rounds fine-tune. This pre-calibration is also why the joint fit below can converge so much tighter than a version that skips straight to `fit_dm_and_offsets` from the untouched library defaults: `mainSlope`/`maskShifts`/`rooftop` already start close to their true values by the time it runs, so the joint optimizer only has to refine them alongside the DM's misregistration, rather than search for all of it at once.

Feel free to run these cells more than once.

In [ ]:
for i in range(6):

    if i % 2 == 0:
        final_loss = calibrator.fit_pupil_to_reference(
            wfs_true.reference_intensity,
            wfs_nominal.parameters(),
            lr=2*10**-(3+i/5),
            n_iter=100,
        )
    else:
        final_loss = calibrator.fit_rooftop(wfs_true.reference_intensity, lr=10**-(1+i/5), n_iter=100)

## Quality of the iMat after the fit of the WFS

If we compare the bench and current calibrated interaction matrices you can observe that both share the same pupil positions. Now is the time to fit the parameters of the DM. You can observe that the illumination pattern of the different modes in the bench and in simulation match in general shape, but not in orientation. This is a classic sign of a problem with the flips (top-bottom and left-right) of the DM and the sign of the influence function (a positive command should correspond to a positive or negative phase).

In [ ]:
calibrator.sanity_check_plot(bench_iMat, M2C, mode_index, idx=4)

To start with the DM, `calibrator.rough_calibrate_dm(bench_iMat, M2C)` performs an automatic selection of the best starting values for flips, rotations, and sign of the DM.

In [ ]:
calibrator.rough_calibrate_dm(bench_iMat, M2C)

## Quality of the fit after the rough calibration

You can observe that the rough calibration found a starting rotation and a flip. The rough calibration rotates the DM in steps of 45 degrees, checking for a good match. It also flips and changes the sign, and returns the best combination of parameters.

In [ ]:
calibrator.sanity_check_plot(bench_iMat, M2C, mode_index, idx=4)

If you find that the rough calibration did not do a good job, you can change those values by hand, like `dm.rotationAngle = 50` and checking again with the `sanity_check_plot`.

## Running the joint fit

`fit_dm_and_offsets` jointly optimizes the DM's misregistration/moffat/sign/mechCoupling parameters, the WFS's mask parameters, and (with `fit_static_offsets=True`) a small reference-pupil/reference-phase offset map, all against the single synthetic `bench_iMat` target. This is the same joint optimizer as before, just with everything upstream of it now in a much better starting position: the pre-calibration step already got `mainSlope`/`maskShifts`/`rooftop` close to their true values, and `rough_calibrate_dm` already found the right rotation quadrant/flip/sign combination -- so this fit is mostly fine-tuning a good starting point rather than searching from scratch, which is why a modest `n_iter=300`/`lr_wfs=3e-3` is enough.

In [ ]:
final_loss, original_positions, transformed_positions = calibrator.fit_dm_and_offsets(
    bench_iMat, M2C, mode_index,
    n_iter=300, lr_dm=1e-2, lr_wfs=3e-3,
    lr_offset_start=-10, lr_offset_end=-2,
    fit_static_offsets=True, plot_mode_idx=4,
)
print(f"Final loss: {final_loss:.4g}")

In [ ]:
calibrator.plot_actuator_and_offsets(original_positions, transformed_positions)

## Did the fit actually recover the ground truth?

This is the entire point of the exercise, and the one thing no real per-instrument `Calibrate<Instrument>Twin.ipynb` notebook can do: compare the *recovered* parameter values directly against the *injected* ones, not just look at whether the loss went down. `dm_nominal.GetMisreg()` returns the DM's misregistration and fittable-parameter values in the same physical units we injected them in; the WFS's `mainSlope`/`rooftop`/`maskShifts` can be read back directly. Don't expect every row below to land on zero error, even once `final_loss` above is very small -- see the note right after the table for why.

In [ ]:
misreg_recovered, dmdict_recovered = dm_nominal.GetMisreg()

comparison = {
    "mainSlope": (1.4, wfs_nominal.mainSlope.item()),
    "rooftop": (0.95, wfs_nominal.rooftop.item()),
}
maskShifts_true = [1.09, 0.90, 0.98, 1.04, 1.05, 0.99, 0.99, 0.91]
maskShifts_recovered = wfs_nominal.maskShifts.detach().cpu().flatten().tolist()
for i, (t, r) in enumerate(zip(maskShifts_true, maskShifts_recovered)):
    comparison[f"maskShifts[{i}]"] = (t, r)

for key, true_val in true_misreg.items():
    comparison[key] = (true_val, misreg_recovered[key])

dm_overrides_true = dict(moffatParam=2.6, signedAmplitude=1.3e-5, MechCoupling=0.42)
for key, true_val in dm_overrides_true.items():
    comparison[key] = (true_val, dmdict_recovered[key])

print(f"{'parameter':<18}{'injected':>14}{'recovered':>14}{'error':>14}")
for key, (true_val, recovered_val) in comparison.items():
    print(f"{key:<18}{true_val:>14.5g}{recovered_val:>14.5g}{true_val - recovered_val:>14.5g}")

### Reading the table: convergence vs. degeneracy

`final_loss` should land very small (well under `1e-3`), and most rows in the table should land close to their injected values -- `rotationAngle`, `shiftX`/`shiftY`, `rooftop`, `signedAmplitude`, `moffatParam`, and `MechCoupling` typically recover to within a fraction of a percent, and `mainSlope`/`maskShifts` to within a few percent.

`radialScaling`, `tangentialScaling`, and `anamorphosisAngle` are the exception worth calling out: expect a visibly larger residual error on these three than on everything else, even though the overall loss is just as small. That's a real degeneracy, not the optimizer failing: rotating the anamorphosis axis by 90 degrees while swapping `radialScaling` and `tangentialScaling` (and flipping their signs) describes very nearly the same physical DM shape, so the fit doesn't have a strong signal telling it which of the two equivalent descriptions to prefer, and can land partway toward the alternative one instead of squarely on the injected values. This is exactly the kind of thing `crosstalk_diagnostic` below is meant to expose, and exactly why a real per-instrument fit's low residual shouldn't be read as "every recovered parameter is individually correct" without checking further.

In [ ]:
labels = list(comparison.keys())
errors = [abs((true_val - recovered_val)/true_val) for true_val, recovered_val in comparison.values()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, errors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel("% error")
ax.set_title("Per-parameter recovery error")
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

## Reconstruction-matrix diagnostics

We can recompute the reconstruction matrix and check again at the cross-talk between the modes in the bench and the simulated one. If the fit was successful, we should observe an image that closely resembles the identity -- compare it to the baseline crosstalk plot near the top of this notebook, from before any calibration.

In [ ]:
modes = calibrator.rebuild_reconstruction_matrix(M2C)
cov = calibrator.crosstalk_diagnostic(bench_iMat)

## Saving the results

`calibrator.save("Tutorials", data_dir="../../Data")` persists the fitted twin -- it delegates to `wfs.SaveCalibration`/`dm.SaveCalibration`, writing `Data/Tutorials/TutorialsWFS.pth` and `Data/Tutorials/TutorialsDM.pth`. `"Tutorials"` here plays the role a real instrument name (`"Rama"`, `"Papyrus"`, ...) plays in the per-instrument calibration notebooks -- it's not tied to any real instrument, just the label this basics series uses for its own synthetic twin. This is the literal hand-off point to `05_TrainingAReconstructor.ipynb`, which loads these same two files via `wfs.LoadCalibration`/`dm.LoadCalibration` from that exact path -- run this cell before switching to notebook 5 if you want it to train against the calibrated twin rather than falling back to uncalibrated defaults.

The cell below demonstrates the round trip: build a fresh `wfs`/`dm` pair from scratch, point the calibrator at them, `calibrator.load(...)` the files just saved, and re-run `rebuild_reconstruction_matrix`/`crosstalk_diagnostic` -- it should reproduce the same crosstalk plot as just above, confirming nothing was lost in the save/load round trip.

In [ ]:
PATH_WFS, PATH_DM = calibrator.save("Tutorials", data_dir="../../Data")

In [ ]:
wfs = PyramidWFS(WFSParams, device)
dm = DeformableMirror(WFSParams,
                    DMParams,
                    device=device)

calibrator.wfs = wfs
calibrator.dm = dm
calibrator.load("Tutorials", data_dir="../../Data")

modes = calibrator.rebuild_reconstruction_matrix(M2C)
cov = calibrator.crosstalk_diagnostic(bench_iMat)

## What this notebook doesn't cover

`fit_static_offsets=True` still fits a reference-pupil/reference-phase offset map above, but the true twin here doesn't inject a deliberately *known* static offset to check that fit against -- and we don't sweep the nominal twin's initial-guess perturbation to characterize `TwinCalibrator`'s basin of convergence (how far off a starting guess can be before the fit stops converging). Both are natural extensions of this exercise, noted as future work in `Ideas/03-ground-truthing-twincalibrator.md`.